In [ ]:
#-------------------------------------------------------------------------------
# Name:        01_check_bbox
# Purpose:     Rasterize the vecor input mask so that they are 512x512px
#
# Author:      Gijs van den Dool & Meggison Oritsemisan
#
# Created:     22/04/2024
# Copyright:   (c) Project Canopy Watch 2024
# Licence:     <Project Canopy Watch>
#-------------------------------------------------------------------------------
# Project Data Folder:
# https://drive.google.com/drive/folders/19wF5gdNnmX4Lwj1kHxltKGRF6DT1yvQ7

# Vector Masks:
# Output from the selection of SB areas in a 5120x5120 bounding box

# Raster Masks:
# Input for the modelling, as a direct conversion from vector to raster

In [ ]:
import geopandas as gpd
import os
from shapely.geometry import Polygon
import shapely.geometry as geometry

# path to the input data
parent_dir = os.path.dirname(os.getcwd())
aoi_path = os.path.join(parent_dir, 'Iter3_93_SB.geojson')

gdf_ISL_lines = gpd.read_file(aoi_path)

print(len(gdf_ISL_lines)) # total lines in the AoI
print(gdf_ISL_lines.crs)  # current projection
gdf_ISL_lines.head(3)


5
EPSG:4326


,feature_id,image_id_list,image_name,time_frame,min_date,max_date,project_id,sentinel_image_url,geometry
0,clp0w4t7600053j6nhf48izpl,Shifting_cultivation_2_0,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.81220 -0.02022, 14.81328 -0.02106..."
1,clp0w5eez00063j6n5rjyqtb7,Shifting_cultivation_2_1,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.82301 -0.03273, 14.82421 -0.03213..."
2,clp0w6bs400083j6n6ll5ehbn,Shifting_cultivation_2_2,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.84184 -0.03270, 14.84328 -0.03175..."


In [ ]:
# gdf_ISL_lines = gdf_ISL_lines.rename(columns={'img_date': 'image_date'})
gdf_ISL_lines = gdf_ISL_lines.rename(columns={'sentinel_image_url': 'url', 'time_frame': 'image_date'})

gdf_ISL_lines

,feature_id,image_id_list,image_name,image_date,min_date,max_date,project_id,url,geometry
0,clp0w4t7600053j6nhf48izpl,Shifting_cultivation_2_0,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.81220 -0.02022, 14.81328 -0.02106..."
1,clp0w5eez00063j6n5rjyqtb7,Shifting_cultivation_2_1,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.82301 -0.03273, 14.82421 -0.03213..."
2,clp0w6bs400083j6n6ll5ehbn,Shifting_cultivation_2_2,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.84184 -0.03270, 14.84328 -0.03175..."
3,clp0w6jl800093j6npmaucis2,Shifting_cultivation_2_3,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.84593 -0.02461, 14.84467 -0.02555..."
4,clp0w6z7g000a3j6nm7qt77m0,Shifting_cultivation_2_4,93_industrial agriculture_shifting cultivation...,2023-02-04,2023-02-03,2023-02-05,clnfxwt64103z07zmfpm263oy,https://apps.sentinel-hub.com/eo-browser/?zoom...,"POLYGON ((14.85101 -0.02951, 14.85339 -0.03180..."


In [ ]:
resent_AoI = []
resent_AoI_small = []
selected_AoI = []

aoi_id = 'SB_93' # specify aoi id for different files

# Modified attr_dict to include new attributes
attr_dict = {
    'geometry': [],
    'image_AoI': [],
    'min_date': [],
    'max_date': [],
    'url': [],
    'image_date': []
}


filtered_df = gdf_ISL_lines

filtered_df.total_bounds

limLngMin=filtered_df.total_bounds[0] #["minx"]
limLatMin=filtered_df.total_bounds[1] #["miny"]
limLngMax=filtered_df.total_bounds[2] #["maxx"]
limLatMax=filtered_df.total_bounds[3] #["maxy"]

#set the bounds:
bound_points_shapely = geometry.MultiPoint([(limLngMin, limLatMin), (limLngMax, limLatMax)])

# get lat/lng from center (below: True centroid - since coords may be multipoint)
input_lon_center = bound_points_shapely.centroid.coords[0][0]
input_lat_center = bound_points_shapely.centroid.coords[0][1]

# set the epsg_code
utm_code=32634
    # define project_from_to with iteration
    # see https://gis.stackexchange.com/a/127432/33092

    # get the bounds
lon_lat_list=((limLngMin,limLatMin ),(limLngMin,limLatMax),(limLngMax,limLatMax),(limLngMax,limLatMin))

polygon_geom = Polygon(lon_lat_list)
polygon = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[polygon_geom]) #careful with the hard coded EPSG

    # Reproject to a system with meters:
polygon_rp = polygon.to_crs(epsg=utm_code)

polygon_width = polygon_rp.bounds.maxx - polygon_rp.bounds.minx
    #print("W: ", polygon_width)

polygon_height = polygon_rp.bounds.maxy - polygon_rp.bounds.miny
    #print("H: ", polygon_height)

    ##
    ### the area should have a minimum of 9 patches meaning extent should be
    ### [3 x 128 x 10 = 3840m] by [3 x 128 x 10 = 3840m]
    ##

f=polygon_rp.centroid
    #print("f is",f[0].coords[0])
centre_coords=f[0].coords[0]

    # creating an empty geometry:
geo_new_polygon=[]

boo_width = False
boo_height = False

    # set the new cell dimentions:
if polygon_width[0]//1280 + 1 < 3:
    cells_w = 3
    boo_width = True
else:
    cells_w = polygon_width[0]//1280 + 1

if polygon_height[0]//1280 + 1 < 3:
    cells_h = 3
    boo_height = True
else:
    cells_h = polygon_height[0]//1280 + 1

    #Set the minimal criteria:
    # set the new cell dimentions:
boo_width2 = True
boo_height2 = True
if polygon_width[0]//1280 >1:
    boo_width2 = False

if polygon_height[0]//1280>1:
    boo_height2 = False

    # correction factor for all directions
factor = ((128 / 2) * 10)

    # adding a buffer of one cell to all directions:
newx1=centre_coords[0] - (factor * cells_w + 10)
newx2=centre_coords[0] + (factor * cells_w + 10)

newy1=centre_coords[1] - (factor * cells_h + 10)
newy2=centre_coords[1] + (factor * cells_h + 10)

new_xy_coords=((newx1,newy1),(newx2,newy1),(newx2,newy2),(newx1,newy2))
new_polygon=Polygon(new_xy_coords)

# The new geometry:
geo_new_polygon.append(new_polygon)

geo_new_polygon

# colors = list(gdf_ISL_lines['color'])[0]

attr_dict['geometry'].append(geo_new_polygon[0])
attr_dict['image_AoI'].append(aoi_id)

# Example appending attributes from the first row of filtered_df,
# you might need to adjust this based on your actual logic for multiple polygons.
attr_dict['min_date'].append(filtered_df['min_date'].iloc[0])
attr_dict['max_date'].append(filtered_df['max_date'].iloc[0])
attr_dict['url'].append(filtered_df['url'].iloc[0])
attr_dict['image_date'].append(filtered_df['image_date'].iloc[0])

    # The new data frame:
polygon_new = gpd.GeoDataFrame(attr_dict,crs="epsg:{}".format(utm_code))

polygon_to_qc=polygon_new.to_crs("epsg:4326")

output_file = aoi_id + ".geojson"
output_file = os.path.join(parent_dir, output_file)
polygon_to_qc.to_file(output_file)

if boo_width and boo_height:
    resent_AoI.append(output_file)
else:
    selected_AoI.append(output_file)

# if boo_width2 and boo_height2:
#     resent_AoI_small.append(output_file)

polygon_to_qc

,geometry,image_AoI,min_date,max_date,url,image_date
0,"POLYGON ((14.80807 -0.03900, 14.86543 -0.03900...",SB_93,2023-02-03,2023-02-05,https://apps.sentinel-hub.com/eo-browser/?zoom...,2023-02-04


In [ ]:
polygon_to_qc.url[0]

'https://apps.sentinel-hub.com/eo-browser/?zoom=13&lat=3.33881&lng=18.07577&themeId=DEFAULT-THEME&visualizationUrl=https%3A%2F%2Fservices.sentinel-hub.com%2Fogc%2Fwms%2Fbd86bcc0-f318-402b-a145-015f85b9427e&datasetId=S2L2A&fromTime=2018-02-02T00%3A00%3A00.000Z&toTime=2018-02-02T23%3A59%3A59.999Z&layerId=1_TRUE_COLOR&demSource3D=%22MAPZEN%22'